# Introduction: Fixed Points as Layers

The entry point is the equation

$$
z^\star=f_\theta(z^\star,x).
$$

The symbol $x$ is the external stimulus and $z^\star$ is the hidden state
that is consistent with the transition $f_\theta$. A finite neural network
layer applies a map once. A deep equilibrium layer asks for the state that would
remain unchanged if the map were applied again.

This notebook follows the introduction theme from the Deep Implicit Layers
tutorial and runs it through the `silva_networks` solver API.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})

from silva_networks import resolve_device, SolverConfig

torch.manual_seed(7)
np.random.seed(7)
device = resolve_device("cuda" if torch.cuda.is_available() else "cpu")
device

## Scalar Warm-Up

Start with a scalar contraction:

$$
f(z)=\tanh(az+b),\qquad |a|<1.
$$

The Picard step with damping is

$$
z_{k+1}=(1-\alpha)z_k+\alpha f(z_k).
$$

The residual at step $k$ is

$$
r_k=|f(z_k)-z_k|.
$$

In [ ]:
from silva_networks import fixed_point, silva_residual_ratio

a = torch.tensor(0.55, device=device)
b = torch.tensor(0.25, device=device)
z0 = torch.zeros((), device=device)

def scalar_f(z):
    return torch.tanh(a * z + b)

configs = [
    SolverConfig(solver="picard", max_iter=25, alpha=0.8),
    SolverConfig(solver="anderson", max_iter=12, alpha=0.8, history=4),
    SolverConfig(solver="broyden", max_iter=12, alpha=0.8),
]

solves = {cfg.solver: fixed_point(scalar_f, z0, cfg) for cfg in configs}
[(name, float(result.z.detach().cpu()), result.iterations, result.residual) for name, result in solves.items()]

In [ ]:
plt.figure(figsize=(5.5, 3.2))
for name, result in solves.items():
    plt.plot(result.residuals, marker="o", label=f"{name}, ratio={silva_residual_ratio(result.residuals):.2e}")
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("residual")
plt.legend()
plt.tight_layout()

## Package Fixed-Point Block

The scalar map becomes a vector map by replacing $a$ with $W_z$, $b$ with
an input-dependent term, and solving

$$
z^\star=\tanh(W_z z^\star + W_x x + b).
$$

`silva_fixed_point_block` exposes this equation directly.

In [ ]:
from silva_networks import silva_fixed_point_block

x = torch.randn(8, 3, device=device)
block = silva_fixed_point_block(
    in_dim=3,
    state_dim=6,
    config=SolverConfig(solver="anderson", max_iter=8, alpha=0.7, history=3),
).to(device)

result = block(x, return_result=True)
result.z.shape, result.iterations, result.residuals[:3], result.residual

## A Tiny Classifier

The equilibrium state can feed an ordinary PyTorch head:

$$
\hat y = W_o z^\star + c.
$$

The code below uses a synthetic two-class task so the notebook is quick on CPU
and also runs unchanged on a Colab GPU runtime.

In [ ]:
from silva_networks import silva_fixed_point_classifier

x_train = torch.randn(32, 4, device=device)
y_train = (x_train[:, 0] + 0.5 * x_train[:, 1] > 0).long()
model = silva_fixed_point_classifier(
    in_features=4,
    state_dim=12,
    num_classes=2,
    config=SolverConfig(solver="picard", max_iter=8, alpha=0.6),
).to(device)
optim = torch.optim.Adam(model.parameters(), lr=0.03)

losses = []
for _ in range(8):
    optim.zero_grad()
    logits = model(x_train)
    loss = torch.nn.functional.cross_entropy(logits, y_train)
    loss.backward()
    optim.step()
    losses.append(float(loss.detach().cpu()))

losses[:3], losses[-1]

In [ ]:
plt.figure(figsize=(4.8, 3.0))
plt.plot(losses, marker="o")
plt.xlabel("training step")
plt.ylabel("cross entropy")
plt.tight_layout()

## Citation and Sources

If this package or notebook is used, cite the software repository:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.0.0. MIT License.
https://github.com/jseluis/silva-networks
```

When the work is connected to the SILVA methodology, cite the SILVA Networks
paper as well:

```text
Jose Luis Lima de Jesus Silva. SILVA Networks as Structured Implicit Layers and
Vector Attractors via Dynamic Interaction Fields. 2026. arXiv:2607.28989.
https://arxiv.org/abs/2607.28989
```

Background sources:

- Deep Implicit Layers tutorial: https://implicit-layers-tutorial.org/
- LocusLab DEQ repository: https://github.com/locuslab/deq
- Deep Equilibrium Models: https://arxiv.org/abs/1909.01377
- Multiscale Deep Equilibrium Models: https://arxiv.org/abs/2006.08656
- Stabilizing Equilibrium Models by Jacobian Regularization: https://arxiv.org/abs/2106.14342

The notebook is adapted to the `silva_networks` public API. It links to the
sources above and keep the examples package-native.